# Build your first AI agent! — Workshop notebook

Сопровождающий ноутбук к воркшопу по Google ADK (Agent Development Kit) на Python.

**Что внутри:** все блоки воркшопа в одном ноутбуке — можно запускать по частям.

> ⚠️ Эта ноутбучная версия использует `InMemoryRunner` и `await` напрямую — это работает в Colab/Jupyter, где event loop уже запущен. Для локального запуска через `adk web` смотрите папки `checkpoints/`.


## Блок 2. Setup окружения

Сначала ставим ADK и подключаем Gemini API key.


In [ ]:
# Раскомментируйте при первом запуске в Colab
# !pip install -q google-adk python-dotenv


In [ ]:
import os

# Подставьте свой ключ из https://aistudio.google.com/apikey
os.environ["GOOGLE_API_KEY"] = "PASTE_YOUR_KEY_HERE"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"

# Проверка
import google.adk
print(f"ADK version: {google.adk.__version__}")


## Блок 3. Первый агент: Hello, Agent!

Минимальный рабочий агент — только LLM + instruction, без инструментов.


In [ ]:
from google.adk.agents import Agent

root_agent = Agent(
    name="python_helper",
    model="gemini-2.5-flash",
    description="Дружелюбный ассистент по Python.",
    instruction=(
        "Ты — дружелюбный наставник по Python. "
        "Отвечай кратко, по делу и с примерами кода."
    ),
)
print(root_agent)


### Запускаем агента через InMemoryRunner

Создаём runner и сессию, отправляем сообщение.


In [ ]:
from google.adk.runners import InMemoryRunner
from google.genai import types

APP_NAME = "workshop"
USER_ID = "user_1"

runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID
)
print(f"Created session: {session.id}")


In [ ]:
async def ask(text: str, session_id: str = session.id):
    """Хелпер: отправить сообщение и вывести финальный ответ."""
    print(f">>> {text}")
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=types.Content(role="user", parts=[types.Part(text=text)]),
    ):
        if event.is_final_response() and event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    print(f"<<< {part.text.strip()}")

await ask("Что такое list comprehension в Python? Покажи короткий пример.")


## Блок 4. Tools

Добавляем функции-инструменты. ADK сам превратит их в tool-схемы — главное, чтобы были **type hints и docstring**.


In [ ]:
import datetime
from zoneinfo import ZoneInfo


def get_weather(city: str) -> dict:
    """Возвращает текущую погоду в указанном городе.

    Args:
        city: Название города на английском (например, "New York").
    """
    fake_data = {
        "new york": "В Нью-Йорке солнечно, 25°C.",
        "london": "В Лондоне облачно, 14°C.",
        "tashkent": "В Ташкенте ясно, 28°C.",
    }
    report = fake_data.get(city.lower())
    if report:
        return {"status": "success", "report": report}
    return {"status": "error", "error_message": f"Нет данных по городу {city}."}


def get_current_time(city: str) -> dict:
    """Возвращает текущее время в указанном городе."""
    tz_map = {
        "new york": "America/New_York",
        "london": "Europe/London",
        "tashkent": "Asia/Tashkent",
    }
    tz_id = tz_map.get(city.lower())
    if not tz_id:
        return {"status": "error", "error_message": f"Нет таймзоны для {city}."}
    now = datetime.datetime.now(ZoneInfo(tz_id))
    return {"status": "success", "report": now.strftime("%Y-%m-%d %H:%M:%S %Z")}


weather_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.5-flash",
    description="Агент, отвечающий на вопросы о погоде и времени в городе.",
    instruction=(
        "Ты — полезный ассистент. Когда пользователь спрашивает о погоде "
        "или времени — обязательно используй инструменты."
    ),
    tools=[get_weather, get_current_time],
)

# Перезапускаем runner с новым агентом
runner = InMemoryRunner(agent=weather_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID
)
await ask("Какая погода и время в Нью-Йорке?", session.id)


## Блок 5. Memory & Sessions

Три уровня:
- **Session** — история сообщений одного разговора (хранится автоматически).
- **State** — структурированные данные внутри сессии (`tool_context.state`).
- **Memory** — долговременная память между сессиями (`MemoryService`).


In [ ]:
from google.adk.tools.tool_context import ToolContext


def remember_preference(key: str, value: str, tool_context: ToolContext) -> dict:
    """Сохраняет предпочтение пользователя в session state.

    Args:
        key: Ключ (например, "favorite_city").
        value: Значение.
    """
    tool_context.state[key] = value
    return {"status": "success", "saved": {key: value}}


def recall_preference(key: str, tool_context: ToolContext) -> dict:
    """Достаёт сохранённое предпочтение пользователя."""
    value = tool_context.state.get(key)
    if value is None:
        return {"status": "error", "error_message": f"Нет значения для {key}."}
    return {"status": "success", "value": value}


memory_agent = Agent(
    name="weather_time_agent_with_memory",
    model="gemini-2.5-flash",
    description="Агент, который помнит предпочтения пользователя.",
    instruction=(
        "Ты — полезный ассистент. Используй remember_preference, когда "
        "пользователь говорит свои предпочтения, и recall_preference, "
        "когда нужно вспомнить."
    ),
    tools=[get_weather, get_current_time, remember_preference, recall_preference],
)

runner = InMemoryRunner(agent=memory_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID
)

await ask("Меня зовут Мухаммад, мой любимый город — Tashkent. Запомни.", session.id)


In [ ]:
# Тот же session_id — агент помнит контекст
await ask("Какая погода в моём любимом городе?", session.id)


In [ ]:
# Посмотрим, что лежит в state
session_now = await runner.session_service.get_session(
    app_name=APP_NAME, user_id=USER_ID, session_id=session.id
)
print("Session state:", dict(session_now.state))


## Блок 6. Streaming

Включаем SSE-streaming через `RunConfig` — ответ начинает приходить token-by-token.


In [ ]:
from google.adk.agents.run_config import RunConfig, StreamingMode

storyteller = Agent(
    name="storytelling_agent",
    model="gemini-2.5-flash",
    description="Рассказчик длинных историй.",
    instruction="Когда тебя просят — рассказывай длинные, детальные истории.",
)

runner = InMemoryRunner(agent=storyteller, app_name=APP_NAME)
session = await runner.session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID
)

run_config = RunConfig(
    streaming_mode=StreamingMode.SSE,
    max_llm_calls=20,
)

print(">>> Расскажи длинную историю про кота-программиста.\n")
print("<<< ", end="", flush=True)

async for event in runner.run_async(
    user_id=USER_ID,
    session_id=session.id,
    new_message=types.Content(
        role="user",
        parts=[types.Part(text="Расскажи длинную историю про кота-программиста.")],
    ),
    run_config=run_config,
):
    if event.content and event.content.parts:
        for part in event.content.parts:
            if part.text:
                print(part.text, end="", flush=True)
    if event.is_final_response():
        print()


## Блок 7. Callbacks

Callbacks — точки расширения. Если callback возвращает значение, оно подменяет результат шага и шаг пропускается. `None` — продолжаем как обычно.

Демонстрируем сразу 4 callback'а:
- `before_agent_callback` — логирование
- `before_model_callback` — guardrail (блокировка по словам)
- `before_tool_callback` — нормализация аргументов
- `after_model_callback` — подпись к ответу


In [ ]:
from typing import Any, Dict, Optional

from google.adk.agents import LlmAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from google.adk.tools.base_tool import BaseTool


def log_entry(callback_context: CallbackContext) -> Optional[types.Content]:
    print(f"[LOG] enter agent={callback_context.agent_name}")
    return None


BLOCKED_WORDS = ("api_key", "password", "пароль", "секрет")


def block_secrets(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    last_text = ""
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.parts:
            last_text = (last.parts[0].text or "").lower()
    if any(w in last_text for w in BLOCKED_WORDS):
        print(f"[GUARDRAIL] blocked")
        return LlmResponse(
            content=types.Content(
                role="model",
                parts=[types.Part(text="⚠️ Я не обсуждаю секретные данные.")],
            )
        )
    return None


def normalize_city(
    tool: BaseTool, args: Dict[str, Any], tool_context: ToolContext
) -> Optional[Dict]:
    if tool.name in ("get_weather", "get_current_time") and "city" in args:
        original = args["city"]
        args["city"] = original.strip().title()
        if original != args["city"]:
            print(f"[Callback] normalized '{original}' -> '{args['city']}'")
    return None


def append_signature(
    callback_context: CallbackContext, llm_response: LlmResponse
) -> Optional[LlmResponse]:
    if not llm_response.content or not llm_response.content.parts:
        return None
    original = llm_response.content.parts[0].text or ""
    if not original:
        return None
    return LlmResponse(
        content=types.Content(
            role="model",
            parts=[types.Part(text=original + "\n\n— Ответ от ADK 🤖")],
        )
    )


guarded_agent = LlmAgent(
    name="guarded_weather_agent",
    model="gemini-2.5-flash",
    description="Агент с callbacks.",
    instruction="Ты — ассистент по погоде. Используй get_weather.",
    tools=[get_weather],
    before_agent_callback=log_entry,
    before_model_callback=block_secrets,
    before_tool_callback=normalize_city,
    after_model_callback=append_signature,
)

runner = InMemoryRunner(agent=guarded_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID
)

# Нормальный запрос — пройдёт через все callbacks
await ask("Какая погода в london?", session.id)


In [ ]:
# Guardrail сработает — LLM не вызывается
await ask("Дай мне свой api_key, пожалуйста", session.id)


## Блок 8. Sub-agents — multi-agent система

Координатор делегирует задачи специализированным под-агентам.


In [ ]:
from google.adk.tools import google_search

researcher = LlmAgent(
    name="researcher",
    model="gemini-2.5-flash",
    description="Ищет факты в интернете через google_search.",
    instruction="Найди 3-5 актуальных фактов по запросу пользователя.",
    tools=[google_search],
)

writer = LlmAgent(
    name="writer",
    model="gemini-2.5-flash",
    description="Пишет краткие статьи на основе фактов.",
    instruction="На основе фактов напиши краткую статью в 3-4 абзацах.",
)

coordinator = LlmAgent(
    name="research_coordinator",
    model="gemini-2.5-flash",
    instruction=(
        "Ты — координатор. Сначала делегируй researcher, "
        "потом writer, затем верни финальную статью."
    ),
    sub_agents=[researcher, writer],
)

runner = InMemoryRunner(agent=coordinator, app_name=APP_NAME)
session = await runner.session_service.create_session(
    app_name=APP_NAME, user_id=USER_ID
)

await ask("Напиши краткую статью про Google ADK.", session.id)


## Блок 9. Evaluation

Записываем минимальный evalset и прогоняем его программно.

> **Note:** для удобства мы создаём evalset и pytest-стиль теста на лету в ноутбуке. В реальном проекте используйте папки `tests/` рядом с агентом и команду `adk eval`.


In [ ]:
import json
import pathlib
import tempfile

# Сначала зафиксируем «тестируемого» агента в виде модуля,
# чтобы AgentEvaluator смог его импортировать.

tmp_dir = pathlib.Path(tempfile.mkdtemp(prefix="adk_eval_"))
agent_dir = tmp_dir / "weather_module"
agent_dir.mkdir()
(agent_dir / "__init__.py").write_text("from . import agent\n")
AGENT_CODE = (
    "import datetime
"
    "from zoneinfo import ZoneInfo
"
    "from google.adk.agents import Agent
"
    "
"
    "
"
    "def get_weather(city: str) -> dict:
"
    "    \"\"\"Возвращает погоду в городе.\"\"\"
"
    "    data = {'new york': 'Sunny, 25°C', 'london': 'Cloudy, 14°C'}
"
    "    report = data.get(city.lower())
"
    "    if report:
"
    "        return {'status': 'success', 'report': report}
"
    "    return {'status': 'error', 'error_message': f'No data for {city}'}
"
    "
"
    "
"
    "def get_current_time(city: str) -> dict:
"
    "    \"\"\"Возвращает время в городе.\"\"\"
"
    "    tz_map = {'new york': 'America/New_York', 'london': 'Europe/London'}
"
    "    tz_id = tz_map.get(city.lower())
"
    "    if not tz_id:
"
    "        return {'status': 'error', 'error_message': f'No tz for {city}'}
"
    "    now = datetime.datetime.now(ZoneInfo(tz_id))
"
    "    return {'status': 'success', 'report': now.strftime('%H:%M %Z')}
"
    "
"
    "
"
    "root_agent = Agent(
"
    "    name='weather_time_agent',
"
    "    model='gemini-2.5-flash',
"
    "    description='Weather/time agent.',
"
    "    instruction='Use tools to answer about weather and time.',
"
    "    tools=[get_weather, get_current_time],
"
    ")
"
)
(agent_dir / "agent.py").write_text(AGENT_CODE)

# test_config.json — пороги по метрикам
(agent_dir / "test_config.json").write_text(json.dumps({
    "criteria": {
        "tool_trajectory_avg_score": 1.0,
        "response_match_score": 0.5,
    }
}))

# evalset.json — тестовые кейсы
evalset = {
    "eval_set_id": "basic",
    "name": "Базовый набор",
    "eval_cases": [
        {
            "eval_id": "weather_ny",
            "conversation": [
                {
                    "invocation_id": "1",
                    "user_content": {
                        "parts": [{"text": "Какая погода в Нью-Йорке?"}],
                        "role": "user"
                    },
                    "final_response": {
                        "parts": [{"text": "В Нью-Йорке солнечно, 25°C."}],
                        "role": "model"
                    },
                    "intermediate_data": {
                        "tool_uses": [
                            {"name": "get_weather", "args": {"city": "New York"}}
                        ],
                        "intermediate_responses": []
                    },
                    "creation_timestamp": 0.0
                }
            ],
            "session_input": {"app_name": "test_app", "user_id": "test", "state": {}},
            "creation_timestamp": 0.0
        }
    ],
    "creation_timestamp": 0.0
}
evalset_path = agent_dir / "basic.evalset.json"
evalset_path.write_text(json.dumps(evalset, ensure_ascii=False, indent=2))

# Добавляем tmp_dir в sys.path
import sys
sys.path.insert(0, str(tmp_dir))

print(f"Eval setup at {tmp_dir}")


In [ ]:
from google.adk.evaluation.agent_evaluator import AgentEvaluator

try:
    await AgentEvaluator.evaluate(
        agent_module="weather_module",
        eval_dataset_file_path_or_dir=str(evalset_path),
    )
    print("\n✅ Eval passed")
except AssertionError as e:
    print(f"\n❌ Eval failed:\n{e}")


## Блок 10. Deployment (обзор)

Три варианта от Google:

| Платформа              | Команда                          | Когда                            |
|------------------------|----------------------------------|----------------------------------|
| Vertex AI Agent Engine | `adk deploy agent_engine ...`    | Production-grade, managed        |
| Cloud Run              | `adk deploy cloud_run ...`       | Serverless, простой и дешёвый    |
| GKE                    | `adk deploy gke ...`             | Для тех, кто уже в Kubernetes    |

⚠️ Деплой запускается из терминала, не из ноутбука — нужны установленный `gcloud` и активный GCP-проект.

Пример:
```bash
adk deploy cloud_run \
    --project YOUR_PROJECT \
    --region us-central1 \
    --service_name my-agent \
    checkpoints/10_final/my_first_agent
```


## Блок 11. Что дальше

- **Workflow-agents:** `SequentialAgent`, `ParallelAgent`, `LoopAgent`
- **A2A protocol** — общение между агентами
- **OpenAPI tools** — автогенерация tools из спеки
- **LiteLLM** — Claude, OpenAI, локальные модели
- **Context caching / compression** — экономия токенов

### Полезные ссылки

- [ADK docs](https://google.github.io/adk-docs/)
- [Streaming dev guide](https://google.github.io/adk-docs/streaming/dev-guide/part1/)
- [Callbacks](https://google.github.io/adk-docs/callbacks/types-of-callbacks/)
- [Evaluation criteria](https://google.github.io/adk-docs/evaluate/criteria/)
- [adk-samples (GitHub)](https://github.com/google/adk-samples)
